# Workshop 3 Lab — Advanced
### Computation 101 · IQ Biology Bootcamp 2026

You're about to reconstruct a real RNA-seq analysis in Google Colab — a full Linux computer in your
browser. **Run each cell in order** (Shift+Enter). Where you see `...`, replace it with your code before running.

*Background:* **ZFP36L2** is an RNA-binding protein that grabs certain messenger RNAs and marks them for
destruction. Knock the gene out in a mouse (**KO**), and those mRNAs are no longer destroyed — so they go
**up**. Researchers measured this in six tissues with RNA-seq, then used DESeq2 to score every gene. A gene
counts as **up-regulated** when its `log2FoldChange > 1` *and* `padj < 0.05`.

In [ ]:
# One-time setup. The lines starting with ! are REAL shell commands running in this
# Colab Linux machine — the same commands you'd type on Fiji. This clones the course
# repo (only if it isn't here yet) and points DATA at the ZFP36L2 dataset inside it.
![ -d iqbio-computation-101-2026 ] || git clone --depth 1 https://github.com/gsstephenson/iqbio-computation-101-2026.git
import pandas as pd, os
DATA = 'iqbio-computation-101-2026/data/zfp36l2'
print('data folder contains:', os.listdir(DATA))

## Your job: name it, then prove it's a direct target
You know one gene — `ENSMUSG00000091694` — is up in all six tissues. What is it, and does ZFP36L2
actually bind it (a *direct* target) or is the effect indirect?

In [ ]:
# Rebuild the six up-sets (same as Intermediate).
tissues = ['Lung', 'Liver', 'BM', 'Spleen', 'Ovary', 'Kidney']
up = {t: set(pd.read_csv(os.path.join(DATA, 'de', f'{t}_DE.csv'), index_col=0)
                 .query('log2FoldChange > 1 and padj < 0.05').index) for t in tissues}
GENE = 'ENSMUSG00000091694'

In [ ]:
# The Gene.name column translates the Ensembl ID into a symbol. Look it up in any table.
lung = pd.read_csv(os.path.join(DATA, 'de', 'Lung_DE.csv'), index_col=0)
symbol = lung.loc[GENE, ...]
print(GENE, '=', symbol)

**Apol11b.** Is it a *direct* ZFP36L2 target? The paper attacks this two ways. Genome-wide first: an
**eCLIP** experiment (in MLTC-1 cells) mapped where ZFP36L2 lands on RNA. `derived/eclip_3utr_genes.txt`
holds the genes with a 3′UTR footprint. Intersect them with the up-regulated genes — and check Apol11b.

In [ ]:
eclip = set(open(os.path.join(DATA, 'derived', 'eclip_3utr_genes.txt')).read().split())
print('Apol11b directly bound in its 3′UTR?', GENE in eclip)
# How many up-regulated genes (any tissue) are also directly bound?
up_any = set().union(*up.values())
print('direct targets:', len(up_any & ...))

**Apol11b isn't in the eCLIP set** (`False`) — and that's not a bug, it's in the paper. The eCLIP ran in
one cell line (MLTC-1), where *Apol11b isn't even expressed*, so it couldn't be captured. In the authors'
words: they confirmed ZFP36L2 binds Apol11b's 3′UTR **by gel-shift assay**, 'but surprisingly this gene
was not captured in our eCLIP data set.' The eCLIP still earns its keep genome-wide — here, **17
up-regulated genes** carry a 3′UTR footprint. For Apol11b itself, the proof is its **ARE motif**:
ZFP36L2 recognizes **AU-rich elements** (core `AUUUA`). Count them in a sequence:

In [ ]:
def count_ares(seq):
    return sum(1 for i in range(len(seq) - 4) if seq[i:i+5] == ...)
print(count_ares('CAUUUAUUUAG'))   # expect 2
print(count_ares('UAUUUAU'))       # Apol11b's ARE

The evidence chain, exactly as the paper builds it: **(1)** Apol11b is the *only* gene up in all six
tissues; **(2)** its 3′UTR carries a single 7-mer ARE (`UAUUUAU`), ZFP36L2's recognition motif; **(3)**
ZFP36L2 **directly binds** the Apol11b probe in a **gel-shift assay** (Fig 5C). The eCLIP maps the binding
mode genome-wide but misses Apol11b — a documented limit, since eCLIP only sees genes expressed in its
cell line. Several methods, each partial, converging on one answer. That's how the paper is built.

## One more thing

The dataset you just analyzed isn't a textbook toy. It's a real, published study —
*The RNA binding protein ZFP36L2 displays tissue-selective mRNA targeting in mice*, **RNA Biology (2026)**.

Its first author is **George** — a first-year in your own program, who a year ago sat exactly where
you're sitting now. Everything you just reproduced, a peer generated. That's the whole arc of this
bootcamp: the person one year ahead of you did real science, and today you re-derived their headline
result from their deposited data. Next year, someone reproduces yours.